### 4 Tổng quan về Keras Functional API

Trong TensorFlow, [Functional API](https://www.tensorflow.org/guide/keras/functional) cho phép bạn xây dựng một đồ thị tính toán (computational graph) chứa các layer, trong đó mỗi layer đóng vai trò như một hàm toán học: `output = Layer(...)(input)`. Cấu trúc này mang lại sự linh hoạt vượt trội cho các mô hình phi tuyến tính, mạng có nhánh tắt (residual connection) hay các bài toán đa đầu vào/đa đầu ra.

Quy trình xây dựng đồ thị tính toán bắt đầu từ việc khởi tạo node đầu vào, xâu chuỗi các phép tính và đóng gói thành một đối tượng `Model`:

- **input_img = tf.keras.Input(shape=input_shape):** Khởi tạo điểm đầu vào của đồ thị tính toán. Tham số này quy định kích thước kỳ vọng của mỗi mẫu dữ liệu (không bao gồm batch size).

- **tfl.Conv2D(filters=..., kernel_size=..., padding='same')(input_img):** Áp dụng một tập hợp các bộ lọc tích chập (learnable filters) trượt qua tensor đầu vào.
  - `filters`: Số lượng đặc trưng đầu ra (output channels/feature maps).
  - `kernel_size`: Chiều cao và chiều rộng của cửa sổ tích chập 2D.
  - `padding='same'`: Đệm thêm 0 xung quanh ảnh để kích thước không gian (chiều cao và chiều rộng) được giữ nguyên khi `strides=1`.

- **tfl.ReLU()(Z):** Áp dụng hàm kích hoạt Rectified Linear Unit theo từng phần tử: $\max(0, Z)$.

- **tfl.MaxPool2D(pool_size=(f, f), strides=(s, s), padding='same')(A):** Giảm kích thước không gian (downsampling) bằng cách lấy giá trị lớn nhất trong từng cửa sổ trượt kích thước $(f, f)$ với bước nhảy $(s, s)$.

- **tfl.Flatten()(P):** Trải phẳng (duỗi) tensor nhiều chiều thành một vector 1 chiều cho từng mẫu dữ liệu trong batch.
  - Chuyển đổi tensor có kích thước `(batch_size, h, w, c)` thành `(batch_size, k)`, với $k = h \times w \times c$.

- **tfl.Dense(units=..., activation='softmax')(F):** Lớp kết nối đầy đủ (fully connected), liên kết toàn bộ đầu vào với các nơ-ron đầu ra.
  - `units`: Số lượng nơ-ron ở lớp này (ví dụ: số nhãn phân loại của bài toán).
  - Tự động khởi tạo và cập nhật ma trận trọng số $W$ cùng vector bias $b$ trong suốt quá trình huấn luyện.

- **model = tf.keras.Model(inputs=input_img, outputs=outputs):** Gom toàn bộ đồ thị tính toán từ tensor đầu vào (`inputs`) đến tensor đầu ra (`outputs`) thành một đối tượng mô hình hoàn chỉnh để huấn luyện hoặc dự đoán.

---

#### Thuật ngữ: Window, Kernel, Filter, Pool

- **Kernel vs. Filter:** Trong tích chập 2D, *kernel* là ma trận trọng số 2 chiều trượt qua từng kênh đơn lẻ, còn *filter* là tập hợp toàn bộ các kernel phủ qua mọi kênh đầu vào để tạo ra 1 feature map đầu ra duy nhất.
- **Kernel Size vs. Pool Size:** Cả hai đều chỉ kích thước $(f, f)$ của cửa sổ trượt nơi diễn ra phép toán (`Conv2D` dùng tham số `kernel_size`, `MaxPool2D` dùng `pool_size`).
- **Pool:** Phép toán giảm chiều không gian bằng cách lấy giá trị cực đại (Max) hoặc trung bình (Average) trong từng cửa sổ trượt mà không cần học thêm bất kỳ trọng số nào.

In [1]:
import tensorflow as tf
import tensorflow.keras.layers as tfl

I0000 00:00:1791747865.508842 2808471 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
def VGG16(input_shape = (224,224,3), num_classes = 1000) :
    input_img = tf.keras.Input(shape = input_shape)
    
    #Block 1(2 Conv, 64 filters)
    X = tfl.Conv2D(64 , (3 ,3) , activation = 'relu' , padding = 'same' ,name = 'block1_conv1')(input_img)
    X = tfl.Conv2D(64 , (3 ,3) , activation = 'relu' , padding = 'same' ,name = 'block1_conv2')(X) # mặc định stride= 1 nên không ghi cũng đúng
    X = tfl.MaxPooling2D((2,2),strides = (2,2), name = 'block1_pool')(X)
    
    #Block 2 (2 Conv , 128 filters)
    X = tfl.Conv2D(128 , (3,3), activation = 'relu' , padding ='same',name = 'block2_conv1')(X)
    X = tfl.Conv2D(128 , (3,3), activation = 'relu' , padding ='same',name = 'block2_conv2')(X)
    X = tfl.MaxPooling2D((2,2),strides = (2,2), name = 'block2_pool')(X)
    # Block 3 (3 Conv, 256 filters)
    X = tfl.Conv2D(256 , (3,3), activation = 'relu' , padding ='same',name = 'block3_conv1')(X)
    X = tfl.Conv2D(256 , (3,3), activation = 'relu' , padding ='same',name = 'block3_conv2')(X)
    X = tfl.Conv2D(256 , (3,3), activation = 'relu' , padding ='same',name = 'block3_conv3')(X)
    X = tfl.MaxPooling2D((2,2),strides = (2,2), name = 'block3_pool')(X)   
    # Block 4 (3 Conv, 512 filters)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(X)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(X)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(X)
    X = tfl.MaxPool2D((2, 2), strides=(2, 2), name='block4_pool')(X)

    # Block 5 (3 Conv, 512 filters)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv1')(X)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv2')(X)
    X = tfl.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv3')(X)
    X = tfl.MaxPool2D((2, 2), strides=(2, 2), name='block5_pool')(X)
    
    # Classification Head (3 Dense)
    X = tfl.Flatten(name = 'flatten')(X)
    X = tfl.Dense(4096 , activation = 'relu' , name = 'fc1')(X)
    X = tfl.Dropout(0.5 , name = 'drop1')(X)
    
    X = tfl.Dense(4096 , activation = 'relu' , name = 'fc2')(X)
    X = tfl.Dropout(0.5 , name = 'drop2')(X)
    
    outputs = tfl.Dense(num_classes , activation = 'softmax' , name = 'predictions')(X)
    
    model = tf.keras.Model(inputs = input_img , outputs = outputs , name = "VGG16")
    return model

model = VGG16(input_shape=(224, 224, 3), num_classes=1000)
model.summary()

Model: "VGG16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,00

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)